# Yoruba Exrension

Pilot study: Deployment-Realistic Compression for Low-Resource NMT.

**What this notebook does:** [One sentence]
**Runtime:** [X hours on T4×2 / CPU-only / etc.]
**Outputs:** [Which CSV files it produces, in which folder]
**Dependencies:** See `requirements.txt` in repo root.

For context, see the project [README](../README.md) and [technical report](../writeup/technical_report.md).

In [ ]:
# auto-gptq has no build for Python 3.13, and its failure aborted the WHOLE install,
# so none of the old pins were applied. Use Kaggle's preinstalled transformers/datasets
# and only add what's missing.
!pip install -q -U bitsandbytes accelerate evaluate sacrebleu sentencepiece

import torch, transformers, datasets
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"transformers {transformers.__version__} | datasets {datasets.__version__}")

In [ ]:
import os
import gc
from pathlib import Path
from datetime import datetime

BASE = Path('/kaggle/working/pilot')
for sub in ['notebooks', 'results', 'logs']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H%M')
print(f"Run ID: {RUN_ID}")

### Hugging Face login
1. Accept the terms at https://huggingface.co/datasets/openlanguagedata/flores_plus
2. Create a **read** token at https://huggingface.co/settings/tokens
3. In Kaggle: *Add-ons → Secrets* → add `HF_TOKEN`, and tick it for this notebook. Internet must be ON.
4. Also accept the terms at https://huggingface.co/CohereForAI/aya-23-8B (it's gated too).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
from datasets import load_dataset

# facebook/flores uses a loading script, which current `datasets` no longer supports.
# FLORES+ is the maintained, parquet-based successor (same sentences, same lang codes).
FLORES = "openlanguagedata/flores_plus"

def load_flores(lang_code, split="devtest"):
    ds = load_dataset(FLORES, lang_code, split=split).sort("id")
    if "text" in ds.column_names:          # FLORES+ calls the column "text"
        ds = ds.rename_column("text", "sentence")
    return ds

flores_yor = load_flores("yor_Latn")
flores_hau = load_flores("hau_Latn")
flores_swh = load_flores("swh_Latn")
flores_eng = load_flores("eng_Latn")

# Sentences must line up row-by-row across languages
for name, ds in [("yor", flores_yor), ("hau", flores_hau), ("swh", flores_swh)]:
    assert ds["id"] == flores_eng["id"], f"{name} not aligned with eng"

print(f"Yoruba: {len(flores_yor)} sentences")
print(f"Sample: {flores_yor[0]['sentence'][:100]}")

In [ ]:
NLLB_LANG_CODES = {
    "hau": "hau_Latn",
    "swh": "swh_Latn",
    "eng": "eng_Latn",
    "yor": "yor_Latn",
}

LANG_NAMES = {"hau": "Hausa", "swh": "Swahili", "yor": "Yoruba", "eng": "English"}

# These dicts were never defined before, so .update() would raise NameError
TRANSLATION_PROMPTS = {}
AYA_TRANSLATION_PROMPTS = {}

for lang in ["hau", "swh", "yor"]:
    for a, b in [(lang, "eng"), ("eng", lang)]:
        A, B = LANG_NAMES[a], LANG_NAMES[b]
        TRANSLATION_PROMPTS[(a, b)] = (
            f"Translate the following {A} sentence to {B}. "
            f"Output only the translation, nothing else.\n\n{A}: {{src}}\n\n{B}:"
        )
        AYA_TRANSLATION_PROMPTS[(a, b)] = f"Translate from {A} to {B}: {{src}}"

print(TRANSLATION_PROMPTS[("yor", "eng")].format(src="Ẹ káàárọ̀"))

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList,
)

MODELS = {
    "qwen-1.5b": {"name": "Qwen/Qwen2.5-1.5B-Instruct",       "arch": "causal",  "params_b": 1.5, "batch_size": 32},
    "aya-8b":    {"name": "CohereForAI/aya-23-8B",            "arch": "causal",  "params_b": 8.0, "batch_size": 16},
    "nllb-1.3b": {"name": "facebook/nllb-200-distilled-1.3B", "arch": "seq2seq", "params_b": 1.4, "batch_size": 32},
}

In [ ]:
# Natively-quantised checkpoints (optional, NOT part of the sweep below).
# auto-gptq is dead on Python 3.13 / transformers 5, so the old install cell was removed.
# To use these, run:  !pip install -q gptqmodel   (GPTQ)  — then call load_model("qwen-1.5b-gptq").
MODELS["qwen-1.5b-gptq"] = {
    "name": "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4",
    "arch": "causal",
    "batch_size": 32,
    "native_quant": "gptq-4bit",  # Already quantised at the Hub
}
MODELS["qwen-1.5b-awq"] = {
    "name": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "arch": "causal",
    "batch_size": 32,
    "native_quant": "awq-4bit",
}

In [ ]:
def free_gpu():
    """Clear GPU memory before loading the next model.

    Note: `del model` inside a helper only deletes the helper's local name — the caller's
    reference keeps the model alive. So the caller must `del model, tokenizer` first,
    then call this.
    """
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
def load_model(model_key, precision="fp16"):
    """Load specified model. Handles causal/seq2seq and natively-quantised checkpoints."""
    cfg = MODELS[model_key]
    model_name = cfg["name"]
    arch = cfg["arch"]
    native_quant = cfg.get("native_quant", None)

    # Decoder-only models need LEFT padding for batched generation; NLLB's encoder wants right.
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, padding_side="left" if arch == "causal" else "right"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_cls = AutoModelForCausalLM if arch == "causal" else AutoModelForSeq2SeqLM

    # transformers 5 uses `dtype=`; the old `torch_dtype=` was being ignored, so "fp16"
    # runs were actually loading in fp32 (NLLB-1.3B showed 5.5 GB instead of ~2.7 GB).
    if native_quant is not None:
        # These checkpoints carry their own quantization_config in the model files
        model = model_cls.from_pretrained(model_name, device_map="auto", dtype=torch.float16)
        print(f"  Loaded {model_name} (native {native_quant})")
    elif precision == "fp16":
        model = model_cls.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    elif precision == "int8":
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
        model = model_cls.from_pretrained(
            model_name, quantization_config=bnb_cfg, device_map="auto",
        )
    elif precision == "int4":
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
        )
        model = model_cls.from_pretrained(
            model_name, quantization_config=bnb_cfg, device_map="auto",
        )
    else:
        raise ValueError(f"Unknown precision: {precision}")

    model.eval()
    total_mem = sum(
        torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / 1e9
    print(f"  GPU memory: {total_mem:.2f} GB")
    return model, tokenizer

In [ ]:
from tqdm.auto import tqdm

MAX_NEW_TOKENS = 256

@torch.inference_mode()
def translate_batch(model, tokenizer, sents, src_lang, tgt_lang, model_key, batch_size=8):
    """Translate a list of sentences. Was called but never defined in the original notebook."""
    arch = MODELS[model_key]["arch"]
    hyps = []
    for i in tqdm(range(0, len(sents), batch_size), desc=f"{src_lang}→{tgt_lang}", leave=False):
        chunk = sents[i:i + batch_size]

        if arch == "seq2seq":   # NLLB: set source lang, force target-lang BOS token
            tokenizer.src_lang = NLLB_LANG_CODES[src_lang]
            enc = tokenizer(chunk, return_tensors="pt", padding=True,
                            truncation=True, max_length=512).to(model.device)
            out = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(NLLB_LANG_CODES[tgt_lang]),
                max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False,
            )
            hyps.extend(t.strip() for t in tokenizer.batch_decode(out, skip_special_tokens=True))

        else:                   # Chat LLMs: prompt via chat template, greedy decode
            table = AYA_TRANSLATION_PROMPTS if model_key.startswith("aya") else TRANSLATION_PROMPTS
            template = table[(src_lang, tgt_lang)]
            texts = [
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": template.format(src=s)}],
                    tokenize=False, add_generation_prompt=True,
                )
                for s in chunk
            ]
            enc = tokenizer(texts, return_tensors="pt", padding=True,
                            add_special_tokens=False).to(model.device)
            out = model.generate(
                **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                temperature=None, top_p=None, top_k=None,
                pad_token_id=tokenizer.pad_token_id,
            )
            new_tokens = out[:, enc["input_ids"].shape[1]:]   # drop the prompt
            for t in tokenizer.batch_decode(new_tokens, skip_special_tokens=True):
                lines = [l.strip() for l in t.splitlines() if l.strip()]
                hyps.append(lines[0] if lines else "")       # keep first line only
    return hyps

In [ ]:
import sacrebleu
import time
import pandas as pd

def evaluate_direction(model, tokenizer, src_data, tgt_data, src_lang, tgt_lang,
                       precision, model_key, n_samples=200, batch_size=None):
    """Run one translation direction and compute metrics."""
    src_sents = [x['sentence'] for x in src_data.select(range(n_samples))]
    tgt_refs = [x['sentence'] for x in tgt_data.select(range(n_samples))]

    if batch_size is None:
        batch_size = MODELS[model_key].get("batch_size", 8)

    print(f"\n→ {src_lang}→{tgt_lang} @ {precision}, n={n_samples}")
    t0 = time.time()
    hypotheses = translate_batch(model, tokenizer, src_sents, src_lang, tgt_lang,
                                 model_key=model_key, batch_size=batch_size)
    elapsed = time.time() - t0

    bleu = sacrebleu.corpus_bleu(hypotheses, [tgt_refs]).score
    chrf = sacrebleu.corpus_chrf(hypotheses, [tgt_refs]).score

    result = {
        'run_id': RUN_ID,
        'model': model_key,
        'precision': precision,
        'src_lang': src_lang,
        'tgt_lang': tgt_lang,
        'n_samples': n_samples,
        'bleu': round(bleu, 2),
        'chrf': round(chrf, 2),
        'total_seconds': round(elapsed, 1),
        'sec_per_sentence': round(elapsed / n_samples, 3),
        'gpu_mem_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }
    print(f"  BLEU: {result['bleu']}, chrF: {result['chrf']}, time: {result['total_seconds']}s")

    # Save hypotheses for later inspection
    hyp_path = BASE / 'results' / f'hyps_{model_key}_{src_lang}-{tgt_lang}_{precision}_{RUN_ID}.txt'
    with open(hyp_path, 'w', encoding='utf-8') as f:
        for s, h, r in zip(src_sents, hypotheses, tgt_refs):
            f.write(f"SRC: {s}\nHYP: {h}\nREF: {r}\n\n")

    return result

In [ ]:
# ============================================================
# SESSION A: Yoruba sweep across all three models
# ============================================================

YOR_DIRECTIONS = [
    ('yor', 'eng', flores_yor, flores_eng),
    ('eng', 'yor', flores_eng, flores_yor),
]

MODEL_PRECISION_SWEEP_YOR = {
    "nllb-1.3b": ["fp16", "int8", "int4"],   # Run first — fastest, highest signal
    "qwen-1.5b": ["fp16", "int8", "int4"],
    # NOTE: aya-8b fp16 is ~16 GB — more than one T4 (15 GB). It will spill to CPU and be
    # extremely slow unless you use Kaggle's "GPU T4 x2". Drop "fp16" here if on a single T4.
    "aya-8b":    ["fp16", "int8", "int4"],   # Run last — slowest
}

N_SAMPLES = 1012   # tip: set to ~50 first for a quick end-to-end dry run
all_results = []

for model_key, precisions in MODEL_PRECISION_SWEEP_YOR.items():
    for precision in precisions:
        print(f"\n{'='*60}\n{model_key} / {precision} / YORUBA\n{'='*60}")
        torch.cuda.reset_peak_memory_stats()
        try:
            model, tokenizer = load_model(model_key, precision)
        except Exception as e:
            print(f"  ✗ Failed to load: {e}")
            continue

        for src_lang, tgt_lang, src_data, tgt_data in YOR_DIRECTIONS:
            try:
                result = evaluate_direction(
                    model, tokenizer, src_data, tgt_data,
                    src_lang, tgt_lang, precision,
                    model_key=model_key, n_samples=N_SAMPLES
                )
                all_results.append(result)
                pd.DataFrame(all_results).to_csv(
                    BASE / 'results' / f'yoruba_{RUN_ID}_partial.csv', index=False
                )
            except Exception as e:
                print(f"  ✗ Failed on {src_lang}->{tgt_lang}: {e}")

        del model, tokenizer
        free_gpu()

pd.DataFrame(all_results).to_csv(BASE / 'results' / f'yoruba_full_{RUN_ID}.csv', index=False)
print(f"\n✓ Yoruba sweep saved")